# MIE312 Lab 4 - Impulse Turbines

Developed by Prof. Sullivan (pierre.sullivan@utoronto.ca). Suggestions for improvements are welcome.

This notebook follows the Fall 2026 Lab 4 manual. It keeps the **brake-drum radius** and **effective blade radius** separate, uses the measured pressure and flow rate at each operating point, and calculates torque, jet velocity, water power, brake power, experimental efficiency, theoretical efficiency, and the peripheral velocity factor $\phi$.

## How to use in Colab
1. Open this notebook from the link in the lab manual and choose **File → Save a copy in Drive**.
2. Run the setup cells.
3. Replace each `np.nan` in the Section A tables with your readings, one row per operating point.
4. Run the analysis sections in order.
5. Run Section F to download the figures and tables for your report.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from math import pi

plt.rcParams["figure.figsize"] = (8, 5)
pd.set_option("display.precision", 4)
from IPython.display import display

# Folder for the figures and tables saved for your report (emptied each time this cell runs)
import shutil
from pathlib import Path
OUT_DIR = Path("lab4_outputs")
shutil.rmtree(OUT_DIR, ignore_errors=True)
OUT_DIR.mkdir()

def _has_finite_data(fig):
    """True if any line, marker or bar in the figure has a finite value."""
    for ax in fig.axes:
        for line in ax.get_lines():
            if np.isfinite(np.asarray(line.get_ydata(), float)).any():
                return True
        for coll in ax.collections:
            offsets = np.asarray(coll.get_offsets(), float)
            if offsets.size and np.isfinite(offsets).all(axis=1).any():
                return True
        for patch in ax.patches:
            if np.isfinite(patch.get_height()) and patch.get_height() != 0:
                return True
    return False

def save_fig(name):
    """Save the current figure as a 300-dpi PNG for your report (called just before plt.show())."""
    fig = plt.gcf()
    if not _has_finite_data(fig):
        return  # nothing plotted yet, so don't save an empty figure
    if globals().get("USE_EXAMPLE_DATA", False):
        fig.text(0.5, 0.5, "EXAMPLE DATA", fontsize=48, color="red", alpha=0.2,
                 ha="center", va="center", rotation=30, transform=fig.transFigure)
    fig.savefig(OUT_DIR / f"{name}.png", dpi=300, bbox_inches="tight")

In [ ]:
# Fixed parameters from the lab manual
rho = 1000.0         # kg/m^3 (water; same value as the Lab 1 and Lab 2 notebooks)
Cv = 0.95
r_drum = 0.025       # m, brake-drum radius used only for torque
r_blade = 0.046      # m, effective turbine radius used for blade speed and phi


def kPa_to_Pa(P_kPa):
    return 1000.0 * P_kPa


def L_min_to_m3_s(Q_L_min):
    return Q_L_min / 1000.0 / 60.0

print(f'Brake-drum radius R_d = {r_drum:.3f} m')
print(f'Effective blade radius r_b = {r_blade:.3f} m')

## Section A — Enter your measurements
Enter the **net brake-force difference** $F=B-A$, turbine speed, gauge pressure, and flow rate for each operating point. The unloaded row should have $F=0$ but a nonzero (typically maximum) rotational speed.

The tables follow the manual: 15 operating points for each turbine (row 1 is the unloaded point) and three tachometer–stroboscope comparisons. Delete unused rows or add more as needed.

In [ ]:
# Enter your measurements: replace each np.nan with your reading.
# Record the pressure and flow rate at each operating point, together with the load and speed.
# Columns: trial, net brake force F = B - A (N), speed (rpm), gauge pressure (kPa), flow rate (L/min)

beta90 = pd.DataFrame({       # flat blades
    'trial': list(range(1, 16)),
    'F_N': [np.nan] * 15,
    'omega_rpm': [np.nan] * 15,
    'P_gauge_kPa': [np.nan] * 15,
    'Q_L_min': [np.nan] * 15,
})

beta160 = pd.DataFrame({      # bucket blades
    'trial': list(range(1, 16)),
    'F_N': [np.nan] * 15,
    'omega_rpm': [np.nan] * 15,
    'P_gauge_kPa': [np.nan] * 15,
    'Q_L_min': [np.nan] * 15,
})

# Tachometer vs stroboscope comparison
rpm_compare = pd.DataFrame({
    'trial': [1, 2, 3],
    'F_N': [np.nan] * 3,
    'tachometer_rpm': [np.nan] * 3,
    'stroboscope_rpm': [np.nan] * 3,
})

for label, table in [('β = 90°', beta90), ('β = 160°', beta160), ('speed comparison', rpm_compare)]:
    if table.iloc[:, 1:].isna().all().all():
        print(f"No measurements entered yet for {label}: replace the np.nan entries with your readings.")

beta90, beta160, rpm_compare

## Section B — Torque, jet speed, blade speed, power, $\phi$, and efficiencies
For each operating point,

$$T=R_dF,\qquad V_b=r_b\omega,$$

where $R_d=0.025$ m is the brake-drum radius and $r_b=0.046$ m is the effective blade radius. Do not interchange these radii.

In [ ]:
def turbine_results(df, beta_deg):
    out = df.copy()
    if len(out) == 0:
        # Add the result columns so later plotting/table cells remain well defined.
        for col in ['P_gauge_Pa', 'Q_m3_s', 'Vj_m_s', 'omega_rad_s', 'Torque_Nm',
                    'Vb_m_s', 'phi', 'BHP_W', 'water_power_W', 'eta_exp', 'eta_th']:
            out[col] = pd.Series(dtype=float)
        return out

    out['P_gauge_Pa'] = out['P_gauge_kPa'].apply(kPa_to_Pa)
    out['Q_m3_s'] = out['Q_L_min'].apply(L_min_to_m3_s)
    out['Vj_m_s'] = np.where(
        out['P_gauge_Pa'] >= 0,
        Cv * np.sqrt(2 * out['P_gauge_Pa'] / rho),
        np.nan,
    )
    out['omega_rad_s'] = out['omega_rpm'] * 2 * np.pi / 60
    out['Torque_Nm'] = r_drum * out['F_N']
    out['Vb_m_s'] = out['omega_rad_s'] * r_blade
    out['phi'] = np.where(out['Vj_m_s'] > 0, out['Vb_m_s'] / out['Vj_m_s'], np.nan)
    out['BHP_W'] = out['omega_rad_s'] * out['Torque_Nm']
    out['water_power_W'] = 0.5 * rho * out['Vj_m_s']**2 * out['Q_m3_s']
    out['eta_exp'] = np.where(out['water_power_W'] > 0, out['BHP_W'] / out['water_power_W'], np.nan)
    out['eta_th'] = 2 * out['phi'] * (1 - out['phi']) * (1 - np.cos(np.deg2rad(beta_deg)))
    return out

res90 = turbine_results(beta90, 90)
res160 = turbine_results(beta160, 160)

print('β = 90°')
display(res90)
print('\nβ = 160°')
display(res160)

### Check your sample calculation
The manual asks for one complete worked example from raw data to efficiency. This cell prints each step for the first loaded operating point (β = 160° if entered, otherwise β = 90°) so you can check your hand calculation against it.

In [ ]:
def worked_example(res, beta):
    """Print every step from raw data to efficiency for the first loaded operating point."""
    valid = res.dropna(subset=['eta_exp'])
    valid = valid[valid['F_N'] > 0]
    if len(valid) == 0:
        return False
    r = valid.iloc[0]
    print(f"Worked example: β = {beta}°, trial {r['trial']:g}")
    print(f"  Jet velocity   V_j = C_v √(2p/ρ) = {Cv} × √(2 × {r['P_gauge_Pa']:.0f} Pa / {rho:.0f} kg/m³) = {r['Vj_m_s']:.3f} m/s")
    print(f"  Angular speed  ω = {r['omega_rpm']:.0f} rpm × 2π/60 = {r['omega_rad_s']:.2f} rad/s")
    print(f"  Torque         T = R_d F = {r_drum} m × {r['F_N']:.3f} N = {r['Torque_Nm']:.4f} N·m")
    print(f"  Blade speed    V_b = r_b ω = {r_blade} m × {r['omega_rad_s']:.2f} rad/s = {r['Vb_m_s']:.3f} m/s")
    print(f"  Speed ratio    φ = V_b / V_j = {r['phi']:.3f}")
    print(f"  Brake power    P_b = T ω = {r['BHP_W']:.2f} W")
    print(f"  Water power    P_w = ½ ρ V_j² Q = 0.5 × {rho:.0f} × {r['Vj_m_s']:.3f}² × {r['Q_m3_s']:.3e} m³/s = {r['water_power_W']:.2f} W")
    print(f"  Efficiency     η = P_b / P_w = {r['eta_exp']:.3f}")
    print(f"  Theory         η_th = 2φ(1 − φ)(1 − cos β) = {r['eta_th']:.3f}")
    return True

if not (worked_example(res160, 160) or worked_example(res90, 90)):
    print('Enter at least one loaded operating point (F > 0) to see a worked example.')

## Section C — Theoretical and experimental efficiency

In [ ]:
phi_grid = np.linspace(0, 1, 400)
eta90_th_grid = 2 * phi_grid * (1 - phi_grid) * (1 - np.cos(np.deg2rad(90)))
eta160_th_grid = 2 * phi_grid * (1 - phi_grid) * (1 - np.cos(np.deg2rad(160)))

plt.plot(phi_grid, eta90_th_grid, label="Theoretical β=90°")
plt.plot(phi_grid, eta160_th_grid, label="Theoretical β=160°")
if res90['eta_exp'].notna().any():
    plt.scatter(res90["phi"], res90["eta_exp"], label="Experimental β=90°")
if res160['eta_exp'].notna().any():
    plt.scatter(res160["phi"], res160["eta_exp"], label="Experimental β=160°")
plt.xlabel("Peripheral velocity factor φ")
plt.ylabel("Efficiency η")
plt.title("Theoretical and Experimental Turbine Efficiency")
plt.grid(True, alpha=0.3)
plt.legend()
save_fig("C_efficiency_theory_and_experiment")
plt.show()

# Experimental-only plot
if res90['eta_exp'].notna().any() or res160['eta_exp'].notna().any():
    plt.plot(res90["phi"], res90["eta_exp"], marker="o", label="Experimental β=90°")
    plt.plot(res160["phi"], res160["eta_exp"], marker="o", label="Experimental β=160°")
    plt.xlabel("Peripheral velocity factor φ")
    plt.ylabel("Experimental efficiency η")
    plt.title("Experimental Efficiency vs φ")
    plt.grid(True, alpha=0.3)
    plt.legend()
    save_fig("C_efficiency_experimental_only")
    plt.show()

## Section D — Summary

In [ ]:
def summarize(res, beta):
    if res["eta_exp"].notna().sum() == 0:
        return {"beta": beta, "phi_at_max_exp": np.nan, "max_exp_eta": np.nan}
    idx = res["eta_exp"].idxmax()
    return {
        "beta": beta,
        "phi_at_max_exp": float(res.loc[idx, "phi"]),
        "max_exp_eta": float(res.loc[idx, "eta_exp"]),
        "rpm_at_max_exp": float(res.loc[idx, "omega_rpm"]),
        "force_at_max_exp": float(res.loc[idx, "F_N"]),
    }

summary = pd.DataFrame([summarize(res90, 90), summarize(res160, 160)])
display(summary)

# Theoretical max occurs at phi = 0.5 for both blade angles
print("Theoretical maximum efficiency occurs at φ = 0.5 for both turbines.")
print(f"Theoretical η_max for β=90°  = {2*0.5*(1-0.5)*(1-np.cos(np.deg2rad(90))):.3f}")
print(f"Theoretical η_max for β=160° = {2*0.5*(1-0.5)*(1-np.cos(np.deg2rad(160))):.3f}")

if rpm_compare[['tachometer_rpm', 'stroboscope_rpm']].notna().all(axis=1).any():
    rpm_compare = rpm_compare.copy()
    rpm_compare["difference_rpm"] = rpm_compare["stroboscope_rpm"] - rpm_compare["tachometer_rpm"]
    rpm_compare["percent_diff_%"] = np.where(
        rpm_compare["tachometer_rpm"] != 0,
        100 * rpm_compare["difference_rpm"] / rpm_compare["tachometer_rpm"],
        np.nan,
    )
    print("\nTachometer vs stroboscope:")
    display(rpm_compare)
    print(f"Mean absolute difference = {rpm_compare['difference_rpm'].abs().mean():.2f} rpm")

In [ ]:
# Optional text summary generator
valid_summary = summary.dropna(subset=['max_exp_eta', 'phi_at_max_exp'])
for _, row in valid_summary.iterrows():
    print(
        f"β={int(row['beta'])}°: max experimental efficiency = "
        f"{row['max_exp_eta']:.3f} at φ = {row['phi_at_max_exp']:.3f}"
    )

if len(valid_summary) == 0:
    print('Enter turbine measurements before generating the experimental-efficiency summary.')

## Section E — Discussion prompts
These follow the discussion prompts in the lab manual:
- Why does the bucket-shaped blade (β = 160°) perform better than the flat blade (β = 90°)?
- Why do measured efficiencies usually lie below theoretical efficiencies?
- What losses are neglected in the theoretical model?
- Why does efficiency drop when the blade speed is either too small or too large relative to the jet speed?
- How reliable was the stroboscope compared with the tachometer?
- Which measurements (brake force, speed, gauge pressure, or flow rate) contribute most to the uncertainty in η, and why is the relative uncertainty largest at light loads, where F is small?

## Section F — Save figures and tables for your report
Run this cell after the analysis. It saves every figure above as a 300-dpi PNG and every results table as a CSV file, then downloads them together as `lab4_outputs.zip`. If your browser blocks the download, open the **Files** panel on the left and download the zip from there.

In [ ]:
# Results tables to save -- file name: (results table, the measurement table(s) it comes from)
TABLES = {
    "beta90_results": ("res90", "beta90"),
    "beta160_results": ("res160", "beta160"),
    "summary": ("summary", ["beta90", "beta160"]),
    "tachometer_vs_stroboscope": ("rpm_compare", "rpm_compare")
}

saved_tables = []
def has_measurements(df):
    """True if any measurement column (all columns after the first) holds a number."""
    return isinstance(df, pd.DataFrame) and df.iloc[:, 1:].select_dtypes("number").notna().any().any()

for name, (var, sources) in TABLES.items():
    table = globals().get(var)
    sources = [sources] if isinstance(sources, str) else sources
    if isinstance(table, pd.DataFrame) and any(has_measurements(globals().get(s)) for s in sources):
        table.to_csv(OUT_DIR / f"{name}.csv", index=False)
        saved_tables.append(f"{name}.csv")

saved_figures = sorted(p.name for p in OUT_DIR.glob("*.png"))
if globals().get("USE_EXAMPLE_DATA", False):
    print(EXAMPLE_WARNING)
print("Figures:", ", ".join(saved_figures) if saved_figures else "none yet (run the plotting cells first)")
print("Tables: ", ", ".join(saved_tables) if saved_tables else "none yet (enter your measurements first)")

zip_path = shutil.make_archive(str(OUT_DIR), "zip", OUT_DIR)
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print(f"Not running in Colab; outputs saved to {zip_path}")

## Section G — Submission checklist

Before submitting, consider adding:
- your raw data tables from the lab,
- one complete sample calculation from raw data to efficiency, keeping the brake-drum radius $R_d$ and the effective blade radius $r_b$ distinct,
- captions under each figure,
- one short uncertainty paragraph,
- one short conclusion paragraph,
- any TA-specific formatting requirements.

This notebook is designed to save time on repetitive calculations so you can focus on interpretation and discussion. You are still responsible for understanding the equations, checking the inputs, and showing any worked example required by the lab instructions.